# Zero-Shot Prompting with LangChain/OpenAI

### __Step 1: Set up the OpenAI API Key__
- The code imports the necessary libraries.
- The **os** is used for interacting with the operating system, and __openai__ is the library required to work with OpenAI's API.
- If .env file is setup in the project, then os.getenv can be used to access the api_key

In [ ]:
#!pip list


In [ ]:
#Option 2
import openai
import os
from openai import AzureOpenAI
import dotenv
from dotenv import load_dotenv
#Remember to create a .env file in this environment in a folder location
load_dotenv("/content/.env")
#load_dotenv()

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

In [ ]:
deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")

### __Step 2: Define a Function to Get Completion__
The __get_completion__ function is responsible for sending a prompt to the OpenAI model and receiving its response.

__Parameters:__
  - __prompt__: It is the text input for which the model will generate a completion.
  -  __model__: The gpt-x (whichever is deployed at the endpoint) model is used to perform the tasks.

The __client.chat.completions.create__ function is used to send a request to the API endpoint.
- This request includes the model, the input messages (formatted as a list of dictionaries with user roles and content), and a temperature setting.

In [ ]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI.

    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.

    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.2,
            max_tokens=512
        )

        return response.model_dump()  # Return the full response as dict

    except Exception as e:
        return {"error": str(e)}


**Role** --> Defining who is speaking

    System : Sets the behaviour of the model
    User : Who inputs the data or instruction
    Assistant : The models's Response

Content : The actual text or instruction which is passed by us.

### __Step 3: Define Your Prompt__
- The prompt variable is defined with a simple translation task.

In [ ]:
prompt = "Explain human intelligence with respect to cognitive ability. \
Give me Reasoning behind the response as Reasoning: \
and give me citations as citations: "
response = get_completion(prompt)


In [ ]:
type(response)

dict

In [ ]:
response.keys()

dict_keys(['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage', 'prompt_filter_results'])

In [ ]:
response['choices'][0]['message']['content']

In [ ]:
# Ensure it's a dictionary
if isinstance(response, dict) and "choices" in response:
    print(response["choices"][0]["message"]["content"])
else:
    print("Error:", response)

In [ ]:
response['choices'][0]['message']

{'content': '**Human Intelligence with Respect to Cognitive Ability:**\n\nHuman intelligence refers to the capacity to learn, understand, and apply knowledge, solve problems, and adapt to new situations. Cognitive abilities are the mental skills that underpin these processes, including memory, attention, reasoning, problem-solving, and language. Intelligence is often measured through cognitive tasks that assess these abilities, such as IQ tests. Cognitive ability is thus a core component of human intelligence, enabling individuals to process information, make decisions, and exhibit adaptive behaviors.\n\n**Reasoning:**  \nHuman intelligence is fundamentally linked to cognitive abilities because intelligence is operationalized through the performance of cognitive tasks. Cognitive abilities such as working memory, processing speed, and executive function are essential for reasoning, learning, and problem-solving. Research in psychology and neuroscience shows that individual differences i

In [ ]:
response['choices']

In [ ]:
response['choices'][0]['message']['content']

In [ ]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI with a preloaded conversation.
    """
    messages = [
        {"role": "system", "content": "You are a math tutor"},
        {"role": "user", "content": "Explain me the ..."},
        {"role": "assistant", "content": "The concept is like..."},
        {"role": "user", "content": prompt},   # <-- use the function prompt here
    ]

    try:
        response = client.chat.completions.create(
            model=deployment_name,   # <-- deployment name, not raw model name
            messages=messages,
            temperature=0.9,
            top_p=0.8,
            max_tokens=512

        )
        #return response.model_dump()
        return response.choices[0].message.content   # <-- just return the text
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
reply = get_completion("Give me an example of probability in daily life")
print(reply)

In [ ]:
prompt = "explain trignometry"

response = get_completion(prompt)
print(response)

In [ ]:
#Option 3
#!pip install --upgrade "langchain>=0.3.29" "langchain-core>=1.0.0" langchain-openai langchain_community



In [ ]:
import langchain
print(langchain.__version__)

1.2.14


In [ ]:
#from openai import AzureOpenAI
#from langchain_openai import AzureOpenAI
#from langchain_openai import AzureChatOpenAI

In [ ]:
#LangChain does not use the Responses API
#LangChain does use Chat Completions

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("/content/.env")

from langchain_openai import AzureChatOpenAI
# Initialize client once
client_lc = AzureChatOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1",
    temperature=0.5,
    max_tokens=512
)

In [ ]:
def get_completion_lc(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

In [ ]:
prompt = "Explain human intelligence with respect to cognitive ability"
response = get_completion_lc(prompt)

print(response)